In [6]:
# Import required packages
import pandas as pd
import numpy as np
import os

In [7]:
# Directories
PROJECT_DIR = r"C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/"

CODE_DIR = os.path.join(PROJECT_DIR, "Code")
DATA_DIR = "C:/Users/skazempour/Documents/StockTwits/dataset/v1/data/"
FIGURES_DIR = os.path.join(PROJECT_DIR, "Figures")
TABLES_DIR = os.path.join(PROJECT_DIR, "Tables")
OUTPUT_DATA_DIR = os.path.join(PROJECT_DIR, "Data")

# File names
INPUT_DATA = os.path.join(DATA_DIR, "aggregated", "aggregated_sentiment_returns.pkl") 

In [8]:
data = pd.read_pickle(INPUT_DATA)

In [4]:
# Prepare features and target
# Target variable
TARGET = 'ar_FF5_1'

# Feature engineering
data['log_n_tweets'] = np.log10(1 + data['n_tweets'])

# Select features
FEATURES = ['log_n_tweets', 'sentiment_score']

# Remove missing values
model_data = data[[TARGET] + FEATURES].dropna()

print(f"Sample size: {len(model_data):,}")
print(f"Target: {TARGET}")
print(f"Features: {FEATURES}")

Sample size: 3,032,691
Target: ar_FF5_1
Features: ['log_n_tweets', 'sentiment_score']


# In-sample neural network

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler

# Standardize features for neural network
scaler = StandardScaler()
X = model_data[FEATURES]
y = model_data[TARGET]

X_scaled = scaler.fit_transform(X)

# Initialize and fit the neural network model
nn_model = MLPRegressor(
    hidden_layer_sizes=(16, 8, 4),
    activation='relu',
    solver='adam',
    alpha=0.0001,
    batch_size='auto',
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=200,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    verbose=False
)

# Fit the model
nn_model.fit(X_scaled, y)

# Generate predictions
y_pred = nn_model.predict(X_scaled)

# Calculate performance metrics
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

# Display results
print(f"In-sample Neural Network Results:")
print(f"R² Score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"\nModel Architecture:")
print(f"  Hidden layers: {nn_model.hidden_layer_sizes}")
print(f"  Number of iterations: {nn_model.n_iter_}")
print(f"  Loss: {nn_model.loss_:.6f}")

In-sample Neural Network Results:
R² Score: 0.0001
RMSE: 0.0549

Model Architecture:
  Hidden layers: (8, 4, 2)
  Number of iterations: 19
  Loss: 0.001527


# Out-of-sample predictions

In [12]:
# Out-of-sample predictions with multiple window types
# Parameters
TRAIN_END_DATE = '2011-12-31'
WINDOW_252 = 252  # One trading year
WINDOW_21 = 21    # One trading month

# Ensure date column is datetime
model_data['date'] = pd.to_datetime(data.loc[model_data.index, 'date'])

# Sort by date
model_data = model_data.sort_values('date')

# Get unique dates
unique_dates = model_data['date'].unique()
unique_dates = pd.DatetimeIndex(unique_dates).sort_values()

# Find the starting prediction date (first date after training window)
train_end = pd.to_datetime(TRAIN_END_DATE)
oos_dates = unique_dates[unique_dates > train_end]

print(f"Training window: {unique_dates[0].strftime('%Y-%m-%d')} to {TRAIN_END_DATE}")
print(f"OOS prediction period: {oos_dates[0].strftime('%Y-%m-%d')} to {oos_dates[-1].strftime('%Y-%m-%d')}")
print(f"Number of OOS dates: {len(oos_dates):,}")

Training window: 2010-06-02 to 2011-12-31
OOS prediction period: 2012-01-03 to 2024-01-03
Number of OOS dates: 2,948


In [13]:
# Initialize storage for predictions
predictions_expanding = []
predictions_rolling_252 = []
predictions_rolling_21 = []

# Neural network parameters
nn_params = {
    'hidden_layer_sizes': (8, 4),
    'activation': 'relu',
    'solver': 'adam',
    'alpha': 0.0001,
    'batch_size': 'auto',
    'learning_rate': 'adaptive',
    'learning_rate_init': 0.001,
    'max_iter': 200,
    'random_state': 42,
    'early_stopping': True,
    'validation_fraction': 0.1,
    'n_iter_no_change': 10,
    'verbose': False
}

# Loop through OOS dates
for i, pred_date in enumerate(oos_dates):
    # Test data: observations on pred_date
    test_mask = model_data['date'] == pred_date
    X_test = model_data.loc[test_mask, FEATURES]
    
    if len(X_test) == 0:
        continue
    
    test_indices = model_data.index[test_mask]
    
    # 1. Expanding window: all data up to pred_date
    train_mask_exp = model_data['date'] < pred_date
    X_train_exp = model_data.loc[train_mask_exp, FEATURES]
    y_train_exp = model_data.loc[train_mask_exp, TARGET]
    
    if len(X_train_exp) > 0:
        # Standardize features
        scaler_exp = StandardScaler()
        X_train_exp_scaled = scaler_exp.fit_transform(X_train_exp)
        X_test_scaled = scaler_exp.transform(X_test)
        
        nn_exp = MLPRegressor(**nn_params)
        nn_exp.fit(X_train_exp_scaled, y_train_exp)
        y_pred_exp = nn_exp.predict(X_test_scaled)
        
        for idx, pred in zip(test_indices, y_pred_exp):
            predictions_expanding.append({
                'date': pred_date,
                'index': idx,
                'pred_expanding': pred
            })
    
    # 2. Rolling 252-day window
    date_idx = unique_dates.get_loc(pred_date)
    if date_idx >= WINDOW_252:
        start_date_252 = unique_dates[date_idx - WINDOW_252]
        train_mask_252 = (model_data['date'] >= start_date_252) & (model_data['date'] < pred_date)
        X_train_252 = model_data.loc[train_mask_252, FEATURES]
        y_train_252 = model_data.loc[train_mask_252, TARGET]
        
        if len(X_train_252) > 0:
            # Standardize features
            scaler_252 = StandardScaler()
            X_train_252_scaled = scaler_252.fit_transform(X_train_252)
            X_test_scaled = scaler_252.transform(X_test)
            
            nn_252 = MLPRegressor(**nn_params)
            nn_252.fit(X_train_252_scaled, y_train_252)
            y_pred_252 = nn_252.predict(X_test_scaled)
            
            for idx, pred in zip(test_indices, y_pred_252):
                predictions_rolling_252.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_rolling_252': pred
                })
    
    # 3. Rolling 21-day window
    if date_idx >= WINDOW_21:
        start_date_21 = unique_dates[date_idx - WINDOW_21]
        train_mask_21 = (model_data['date'] >= start_date_21) & (model_data['date'] < pred_date)
        X_train_21 = model_data.loc[train_mask_21, FEATURES]
        y_train_21 = model_data.loc[train_mask_21, TARGET]
        
        if len(X_train_21) > 0:
            # Standardize features
            scaler_21 = StandardScaler()
            X_train_21_scaled = scaler_21.fit_transform(X_train_21)
            X_test_scaled = scaler_21.transform(X_test)
            
            nn_21 = MLPRegressor(**nn_params)
            nn_21.fit(X_train_21_scaled, y_train_21)
            y_pred_21 = nn_21.predict(X_test_scaled)
            
            for idx, pred in zip(test_indices, y_pred_21):
                predictions_rolling_21.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_rolling_21': pred
                })
    
    # Progress update
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(oos_dates)} dates ({100 * (i + 1) / len(oos_dates):.1f}%)")

print(f"\nCompleted.")
print(f"Expanding window predictions: {len(predictions_expanding):,}")
print(f"Rolling 252-day predictions: {len(predictions_rolling_252):,}")
print(f"Rolling 21-day predictions: {len(predictions_rolling_21):,}")

c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.wa

Processed 100/2948 dates (3.4%)


c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.wa

Processed 200/2948 dates (6.8%)


c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.wa

Processed 300/2948 dates (10.2%)


c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.wa

Processed 400/2948 dates (13.6%)


c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.wa

Processed 500/2948 dates (17.0%)
Processed 600/2948 dates (20.4%)
Processed 600/2948 dates (20.4%)
Processed 700/2948 dates (23.7%)
Processed 700/2948 dates (23.7%)
Processed 800/2948 dates (27.1%)
Processed 800/2948 dates (27.1%)
Processed 900/2948 dates (30.5%)
Processed 900/2948 dates (30.5%)
Processed 1000/2948 dates (33.9%)
Processed 1000/2948 dates (33.9%)
Processed 1100/2948 dates (37.3%)
Processed 1100/2948 dates (37.3%)
Processed 1200/2948 dates (40.7%)
Processed 1200/2948 dates (40.7%)
Processed 1300/2948 dates (44.1%)
Processed 1300/2948 dates (44.1%)
Processed 1400/2948 dates (47.5%)
Processed 1400/2948 dates (47.5%)
Processed 1500/2948 dates (50.9%)
Processed 1500/2948 dates (50.9%)
Processed 1600/2948 dates (54.3%)
Processed 1600/2948 dates (54.3%)
Processed 1700/2948 dates (57.7%)
Processed 1700/2948 dates (57.7%)
Processed 1800/2948 dates (61.1%)
Processed 1800/2948 dates (61.1%)
Processed 1900/2948 dates (64.5%)
Processed 1900/2948 dates (64.5%)
Processed 2000/2948 dat

In [14]:
# Convert predictions to DataFrames and merge
df_expanding = pd.DataFrame(predictions_expanding)
df_rolling_252 = pd.DataFrame(predictions_rolling_252)
df_rolling_21 = pd.DataFrame(predictions_rolling_21)

# Merge all predictions together
predictions_df = df_expanding.merge(
    df_rolling_252,
    on=['date', 'index'],
    how='outer'
).merge(
    df_rolling_21,
    on=['date', 'index'],
    how='outer'
)

# Add symbol information
predictions_df = predictions_df.merge(
    data[['symbol']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder columns
predictions_df = predictions_df[['date', 'symbol', 'index', 'pred_expanding', 'pred_rolling_252', 'pred_rolling_21']]

# Sort by date and symbol
predictions_df = predictions_df.sort_values(['date', 'symbol']).reset_index(drop=True)

print("Predictions Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_df):,}")
print(f"\nNon-null predictions by window type:")
print(f"  Expanding: {predictions_df['pred_expanding'].notna().sum():,}")
print(f"  Rolling 252-day: {predictions_df['pred_rolling_252'].notna().sum():,}")
print(f"  Rolling 21-day: {predictions_df['pred_rolling_21'].notna().sum():,}")
print(f"\nFirst 10 predictions:")
print(predictions_df.head(10))

Predictions Summary
Total observations: 3,001,267

Non-null predictions by window type:
  Expanding: 3,001,267
  Rolling 252-day: 3,001,267
  Rolling 21-day: 3,001,267

First 10 predictions:
        date symbol  index  pred_expanding  pred_rolling_252  pred_rolling_21
0 2012-01-03   AAPL  31505       -0.005873          0.004065         0.014993
1 2012-01-03    AAV  31757        0.000044         -0.000199        -0.000282
2 2012-01-03   ABVT  31838        0.000044         -0.000199        -0.000282
3 2012-01-03    ACI  31993        0.000044         -0.000199        -0.000282
4 2012-01-03    ACW  32078       -0.007138         -0.000919        -0.009039
5 2012-01-03    AKS  32643        0.000044         -0.000199        -0.000282
6 2012-01-03    ALK  32752        0.000044         -0.000199        -0.000282
7 2012-01-03   ALXN  32881        0.000044         -0.000199        -0.000282
8 2012-01-03     AM  32942       -0.002879          0.001255        -0.001691
9 2012-01-03    AME  33040   

In [15]:
# Save predictions to Data directory
os.makedirs(OUTPUT_DATA_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(OUTPUT_DATA_DIR, "predictions_neural_network.pkl")
predictions_df.to_pickle(OUTPUT_FILE)

print(f"Predictions saved to: {OUTPUT_FILE}")
print(f"File size: {os.path.getsize(OUTPUT_FILE) / (1024**2):.2f} MB")
print(f"Shape: {predictions_df.shape}")

Predictions saved to: C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/Data\predictions_neural_network.pkl
File size: 128.85 MB
Shape: (3001267, 6)
